# NB14 Token standards — Chapter 14

A chapter notebook for *Blockchain for Lawyers: A Reading Through Technology and Economics*. The code is the same as in the text.

- All code here is a **toy implementation** meant to make a principle visible. Do not use it with real assets, real private keys or real transactions.
- No external API is called, and everything runs without a network. Cells that need the bundled snapshot data carry that data inside the cell.
- Standard library only (run and checked on Python 3.13). There is no install cell.
- The outputs are the editor's own run, stored as produced. Running the cells again should give the same values (random seeds are fixed).

In [ ]:
# NB14-1: imitating the ERC-20 ledger — balance and allowance dictionaries, the six functions, the two events, and the approve race condition (for study)
class ERC20Toy:
    """Only the required functions of EIP-20. Failures raise exceptions (the standard's SHOULD throw). Events are written to self.events."""
    def __init__(self, name, symbol):
        self.name, self.symbol = name, symbol             # OPTIONAL items — may or may not be present
        self.balances = {}                                 # "every address is 0 from the start" — read as 0 if not in the dictionary
        self.allowances = {}                               # (owner, spender) -> remaining allowance
        self.events = []
    def totalSupply(self):          return sum(self.balances.values())
    def balanceOf(self, owner):     return self.balances.get(owner, 0)
    def allowance(self, o, s):      return self.allowances.get((o, s), 0)
    def mint(self, to, value):                            # outside the standard — issuance rules are up to the issuer. Only a Transfer event from 0x0 is recommended (SHOULD)
        self.balances[to] = self.balanceOf(to) + value
        self.events.append(("Transfer", "0x0", to, value))
    def transfer(self, caller, to, value):
        if self.balanceOf(caller) < value:
            raise ValueError(f"insufficient balance: {caller} has only {self.balanceOf(caller)}")
        self.balances[caller] -= value
        self.balances[to] = self.balanceOf(to) + value
        self.events.append(("Transfer", caller, to, value))   # emit the event even for a quantity of 0 (MUST)
        return True
    def approve(self, caller, spender, value):
        self.allowances[(caller, spender)] = value            # "calling it again overwrites"
        self.events.append(("Approval", caller, spender, value))
        return True
    def transferFrom(self, caller, frm, to, value):
        if self.allowance(frm, caller) < value:
            raise ValueError(f"insufficient allowance: the amount {frm} allowed {caller} is {self.allowance(frm, caller)}")
        if self.balanceOf(frm) < value:
            raise ValueError("insufficient balance")
        self.allowances[(frm, caller)] -= value
        self.balances[frm] -= value
        self.balances[to] = self.balanceOf(to) + value
        self.events.append(("Transfer", frm, to, value))
        return True

tok = ERC20Toy("Toy Token", "TOY")
tok.mint("Alice", 1000)
tok.transfer("Alice", "Bob", 250)
print("balances — Alice:", tok.balanceOf("Alice"), "Bob:", tok.balanceOf("Bob"), "Carol (an address never seen before):", tok.balanceOf("Carol"), "| total supply:", tok.totalSupply())
try:
    tok.transfer("Bob", "Carol", 300)
except ValueError as e:
    print("Bob → Carol 300:", e)

# approve / transferFrom — the withdraw workflow in which a contract (here 'Exchange') takes tokens on Alice's behalf
tok.approve("Alice", "Exchange", 100)
tok.transferFrom("Exchange", "Alice", "Dave", 60)
print("Exchange withdraws 60 from Alice → Dave:", tok.balanceOf("Dave"), "| remaining allowance:", tok.allowance("Alice", "Exchange"))

# the approve race condition — reenacting, as a sequence, why the standard recommends "set it to 0 first"
tok2 = ERC20Toy("Toy", "TOY"); tok2.mint("Alice", 1000)
tok2.approve("Alice", "Spender", 100)                      # ① Alice: allows Spender 100
# Alice sends approve(Spender, 50), thinking "100 is too much, cut it to 50." But the order in which transactions enter a block is not the order in which they were sent.
tok2.transferFrom("Spender", "Alice", "Spender", 100)      # ② Spender takes out all 100 before Alice's transaction
tok2.approve("Alice", "Spender", 50)                       # ③ then Alice's approve(50) is applied — because it "overwrites," a fresh 50 is opened
tok2.transferFrom("Spender", "Alice", "Spender", 50)       # ④ Spender takes out another 50
print("race condition — Alice meant to allow only 50, but the amount Spender took:", tok2.balanceOf("Spender"), "(= 100 + 50)")
print("number of events:", len(tok2.events), "| last three:", tok2.events[-3:])

balances — Alice: 750 Bob: 250 Carol (an address never seen before): 0 | total supply: 1000
Bob → Carol 300: insufficient balance: Bob has only 250
Exchange withdraws 60 from Alice → Dave: 60 | remaining allowance: 40
race condition — Alice meant to allow only 50, but the amount Spender took: 150 (= 100 + 50)
number of events: 5 | last three: [('Transfer', 'Alice', 'Spender', 100), ('Approval', 'Alice', 'Spender', 50), ('Transfer', 'Alice', 'Spender', 50)]


In [ ]:
# NB14-2: imitating the ERC-721 ledger — number → owner, approvals and operators, the recipient check in safeTransferFrom, a mutable URI (for study)
MAGIC = "onERC721Received-magic"                           # in reality bytes4(keccak256("onERC721Received(address,address,uint256,bytes)")) — computed in NB14-3 below

class Wallet:                                              # imitates a "contract recipient." A contract that is ready to receive returns the magic value
    def __init__(self, ready): self.ready = ready
    def onERC721Received(self, operator, frm, tokenId, data): return MAGIC if self.ready else "something else"

class ERC721Toy:
    def __init__(self):
        self.owner = {}            # tokenId -> owner
        self.approved = {}         # tokenId -> approved address
        self.operators = set()     # (owner, operator)
        self.uri = {}              # tokenId -> URI string (metadata extension, OPTIONAL)
        self.events = []
    def ownerOf(self, tid):
        if tid not in self.owner: raise KeyError(f"invalid NFT {tid}")   # a query about an invalid NFT (one assigned to the zero address, etc.) fails — the gist of the ownerOf comment in the original
        return self.owner[tid]
    def balanceOf(self, who):  return sum(1 for o in self.owner.values() if o == who)
    def mint(self, to, tid, uri):                           # outside the standard — creation rules are up to the issuer
        self.owner[tid] = to; self.uri[tid] = uri; self.events.append(("Transfer", "0x0", to, tid))
    def tokenURI(self, tid):   self.ownerOf(tid); return self.uri[tid]
    def _authorized(self, caller, tid):
        o = self.ownerOf(tid)
        return caller == o or self.approved.get(tid) == caller or (o, caller) in self.operators
    def approve(self, caller, approved, tid):
        o = self.ownerOf(tid)
        if not (caller == o or (o, caller) in self.operators): raise PermissionError("only the owner or an operator can approve")
        self.approved[tid] = approved; self.events.append(("Approval", o, approved, tid))
    def setApprovalForAll(self, caller, operator, ok):
        (self.operators.add if ok else self.operators.discard)((caller, operator))
        self.events.append(("ApprovalForAll", caller, operator, ok))
    def transferFrom(self, caller, frm, to, tid):
        if not self._authorized(caller, tid): raise PermissionError(f"{caller} has no authority to move {tid}")
        if self.owner[tid] != frm: raise ValueError("_from is not the current owner")
        if to == "0x0": raise ValueError("cannot send to the zero address")
        self.owner[tid] = to; self.approved.pop(tid, None)   # "the approval is reset on transfer"
        self.events.append(("Transfer", frm, to, tid))
    def safeTransferFrom(self, caller, frm, to, tid, data=b"", contracts=None):
        self.transferFrom(caller, frm, to, tid)
        if contracts and to in contracts:                      # if the recipient is a contract, ask whether it is ready to receive
            if contracts[to].onERC721Received(caller, frm, tid, data) != MAGIC:
                self.owner[tid] = frm; self.events.pop()       # roll back (in reality the whole transaction reverts)
                raise ValueError(f"recipient {to} did not return the magic value, so the transfer was rolled back")

nft = ERC721Toy()
nft.mint("Alice", 7, "https://example.invalid/meta/7.json")
print("owner of #7:", nft.ownerOf(7), "| tokenURI:", nft.tokenURI(7))
contracts = {"ReadyContract": Wallet(ready=True), "DumbContract": Wallet(ready=False)}
try:
    nft.safeTransferFrom("Alice", "Alice", "DumbContract", 7, contracts=contracts)
except ValueError as e:
    print("safeTransferFrom → contract not ready:", e, "| owner is still", nft.ownerOf(7))
nft.transferFrom("Alice", "Alice", "DumbContract", 7)       # an "unsafe" transfer sends without asking — this is where the standard's capital-letter warning applies
print("transferFrom → same contract: owner", nft.ownerOf(7), "(locked forever if the contract has no code to get it out)")

nft.mint("Bob", 8, "https://example.invalid/meta/8.json")
nft.setApprovalForAll("Bob", "Marketplace", True)           # Bob makes the marketplace contract an operator
nft.transferFrom("Marketplace", "Bob", "Carol", 8)           # the operator moves Bob's token to Carol
print("owner of #8 after the operator's transfer:", nft.ownerOf(8), "| Bob's count:", nft.balanceOf("Bob"), "| Carol:", nft.balanceOf("Carol"))

nft.uri[8] = "https://example.invalid/meta/8-v2.json"       # the issuer changes the URI — the owner slot stays as it is
print("after the URI change — owner:", nft.ownerOf(8), "| tokenURI:", nft.tokenURI(8), "(the chain guarantees the left; the right can change)")

owner of #7: Alice | tokenURI: https://example.invalid/meta/7.json
safeTransferFrom → contract not ready: recipient DumbContract did not return the magic value, so the transfer was rolled back | owner is still Alice
transferFrom → same contract: owner DumbContract (locked forever if the contract has no code to get it out)
owner of #8 after the operator's transfer: Carol | Bob's count: 0 | Carol: 1
after the URI change — owner: Carol | tokenURI: https://example.invalid/meta/8-v2.json (the chain guarantees the left; the right can change)


In [ ]:
# NB14-3: Keccak-256 by hand — a single padding byte separates SHA3-256 from Keccak-256. Then recompute the ERC-165/721 identifiers (for study, not for real use)
import hashlib
from functools import reduce

RC = [0x0000000000000001, 0x0000000000008082, 0x800000000000808A, 0x8000000080008000, 0x000000000000808B, 0x0000000080000001,
      0x8000000080008081, 0x8000000000008009, 0x000000000000008A, 0x0000000000000088, 0x0000000080008009, 0x000000008000000A,
      0x000000008000808B, 0x800000000000008B, 0x8000000000008089, 0x8000000000008003, 0x8000000000008002, 0x8000000000000080,
      0x000000000000800A, 0x800000008000000A, 0x8000000080008081, 0x8000000000008080, 0x0000000080000001, 0x8000000080008008]
ROT = [[0, 36, 3, 41, 18], [1, 44, 10, 45, 2], [62, 6, 43, 15, 61], [28, 55, 25, 21, 56], [27, 20, 39, 8, 14]]   # rotation amounts r[x][y]
M64 = (1 << 64) - 1
def rol(v, n): return ((v << n) | (v >> (64 - n))) & M64 if n else v

def keccak_f(A):                                           # A[x][y]: 5×5 lanes of 64 bits. 24 rounds of θ·ρ·π·χ·ι
    for rc in RC:
        C = [A[x][0] ^ A[x][1] ^ A[x][2] ^ A[x][3] ^ A[x][4] for x in range(5)]
        D = [C[(x - 1) % 5] ^ rol(C[(x + 1) % 5], 1) for x in range(5)]
        A = [[A[x][y] ^ D[x] for y in range(5)] for x in range(5)]                       # θ
        B = [[0] * 5 for _ in range(5)]
        for x in range(5):
            for y in range(5):
                B[y][(2 * x + 3 * y) % 5] = rol(A[x][y], ROT[x][y])                      # ρ, π
        A = [[B[x][y] ^ ((~B[(x + 1) % 5][y]) & B[(x + 2) % 5][y]) for y in range(5)] for x in range(5)]   # χ
        A[0][0] ^= rc                                                                    # ι
    return A

def sponge256(msg: bytes, pad_byte: int) -> bytes:        # capacity 512 bits → rate 1088 bits = 136 bytes, output 256 bits
    rate = 136
    padded = bytearray(msg) + bytearray([pad_byte]) + bytearray((-len(msg) - 2) % rate) + bytearray([0x80])
    if len(msg) % rate == rate - 1:                        # when only one byte is left: fill it with the single byte pad_byte | 0x80
        padded = bytearray(msg) + bytearray([pad_byte | 0x80])
    A = [[0] * 5 for _ in range(5)]
    for off in range(0, len(padded), rate):
        block = padded[off:off + rate]
        for i in range(rate // 8):                          # lane i → (x, y) = (i % 5, i // 5), little-endian
            A[i % 5][i // 5] ^= int.from_bytes(block[8 * i:8 * i + 8], "little")
        A = keccak_f(A)
    return b"".join(A[i % 5][i // 5].to_bytes(8, "little") for i in range(4))

def sha3_256_manual(m): return sponge256(m, 0x06)          # NIST SHA-3: padding suffix 0x06
def keccak256(m):       return sponge256(m, 0x01)          # Ethereum Keccak-256: padding suffix 0x01

for m in [b"", b"abc", b"a" * 200]:
    print(f"{m[:12]!r:16} by-hand SHA3-256 == hashlib.sha3_256 ? {sha3_256_manual(m) == hashlib.sha3_256(m).digest()}")
print("keccak256(b'')    =", keccak256(b"").hex())
print("sha3_256(b'')     =", hashlib.sha3_256(b"").hexdigest(), "(only one padding byte differs, yet the value is entirely different)")

def selector(sig: str) -> int: return int.from_bytes(keccak256(sig.encode())[:4], "big")
print()
print("transfer(address,uint256) selector =", f"0x{selector('transfer(address,uint256)'):08x}")
print("supportsInterface(bytes4) selector =", f"0x{selector('supportsInterface(bytes4)'):08x}", "(ERC-165 uses this value as the identifier)")

erc721 = ["balanceOf(address)", "ownerOf(uint256)", "safeTransferFrom(address,address,uint256,bytes)",
          "safeTransferFrom(address,address,uint256)", "transferFrom(address,address,uint256)", "approve(address,uint256)",
          "setApprovalForAll(address,bool)", "getApproved(uint256)", "isApprovedForAll(address,address)"]
erc721_meta = ["name()", "symbol()", "tokenURI(uint256)"]
xor = lambda sigs: reduce(lambda a, b: a ^ b, (selector(s) for s in sigs))
print(f"ERC-721 identifier (XOR of 9 functions)    = 0x{xor(erc721):08x}   standard's text: 0x80ac58cd → match {xor(erc721) == 0x80ac58cd}")
print(f"ERC-721 Metadata identifier (3 functions)  = 0x{xor(erc721_meta):08x}   standard's text: 0x5b5e139f → match {xor(erc721_meta) == 0x5b5e139f}")
print(f"onERC721Received magic value               = 0x{selector('onERC721Received(address,address,uint256,bytes)'):08x}   standard's text: 0x150b7a02 → match {selector('onERC721Received(address,address,uint256,bytes)') == 0x150b7a02}")

b''              by-hand SHA3-256 == hashlib.sha3_256 ? True
b'abc'           by-hand SHA3-256 == hashlib.sha3_256 ? True
b'aaaaaaaaaaaa'  by-hand SHA3-256 == hashlib.sha3_256 ? True
keccak256(b'')    = c5d2460186f7233c927e7db2dcc703c0e500b653ca82273b7bfad8045d85a470
sha3_256(b'')     = a7ffc6f8bf1ed76651c14756a061d662f580ff4de43b49fa82d80a4b80f8434a (only one padding byte differs, yet the value is entirely different)

transfer(address,uint256) selector = 0xa9059cbb
supportsInterface(bytes4) selector = 0x01ffc9a7 (ERC-165 uses this value as the identifier)
ERC-721 identifier (XOR of 9 functions)    = 0x80ac58cd   standard's text: 0x80ac58cd → match True
ERC-721 Metadata identifier (3 functions)  = 0x5b5e139f   standard's text: 0x5b5e139f → match True
onERC721Received magic value               = 0x150b7a02   standard's text: 0x150b7a02 → match True
